# Synthetic scatterplot generation and labelling workflow

This notebook implements the **three-knob framework** for constructing labelled synthetic scatterplots.

It focuses only on:

1. generating synthetic scatterplots;
2. deriving ground-truth labels from generation parameters and numerical curve properties;
3. saving case metadata and optional realization data.

It does **not** calculate relationship metrics.

The full design contains 21 function families, 20 shape configurations per family, 50 SNR levels, and two optional interference experiments for noise structure and sampling distribution.


In [ ]:
# ============================================================
# 0. Imports and global settings
# ============================================================

from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 88
N = 500
R = 100
OUTPUT_DIR = Path('./scatterplot_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 50 SNR levels used in the main experiment.
# np.inf means no noise.
SNR_LEVELS = [
    0.05, 0.08, 0.1, 0.13, 0.17, 0.2, 0.25, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0, 2.5, 3.0,
    3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 10, 12, 15, 18,
    20, 25, 30, 35, 40, 50, 60, 80, 100, 130,
    170, 200, 300, 500, 700, 1000, 2000, 5000, 10000, np.inf
]

INTERFERENCE_SNR_LEVELS = [0.5, 1, 2, 5, 10, 20, 50, 100, 500, np.inf]

print('Output directory:', OUTPUT_DIR.resolve())
print('Number of SNR levels:', len(SNR_LEVELS))


## 1. Utility functions

The utilities below handle normalization, numerical derivative-based labels, noise scaling by SNR, and x sampling distributions.


In [ ]:
# ============================================================
# 1. Utility functions
# ============================================================

def safe_range(y: np.ndarray, eps: float = 1e-12) -> float:
    y = np.asarray(y, dtype=float)
    yr = float(np.nanmax(y) - np.nanmin(y))
    return max(yr, eps)


def normalize_range(y: np.ndarray, low: float = 0.0, high: float = 1.0) -> np.ndarray:
    """Normalize y to [low, high]. If y is constant, return midpoint."""
    y = np.asarray(y, dtype=float)
    yr = np.nanmax(y) - np.nanmin(y)
    if not np.isfinite(yr) or yr < 1e-12:
        return np.full_like(y, (low + high) / 2.0, dtype=float)
    return low + (y - np.nanmin(y)) / yr * (high - low)


def linear_fit_residual_ratio(x: np.ndarray, y: np.ndarray) -> float:
    """max absolute residual from a linear fit, divided by y range."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    coef = np.polyfit(x, y, deg=1)
    yhat = np.polyval(coef, x)
    return float(np.nanmax(np.abs(y - yhat)) / safe_range(y))


def derivative_labels(x_grid: np.ndarray, y_grid: np.ndarray, tol: float = 1e-6) -> Dict[str, Any]:
    """Derive labels from numerical first and second derivatives."""
    x_grid = np.asarray(x_grid, dtype=float)
    y_grid = np.asarray(y_grid, dtype=float)

    dy = np.gradient(y_grid, x_grid)
    ddy = np.gradient(dy, x_grid)

    # Ignore tiny numerical derivatives relative to derivative scale.
    dy_scale = max(float(np.nanmax(np.abs(dy))), 1e-12)
    sign_tol = max(tol, 1e-4 * dy_scale)
    dy_sign = np.where(dy > sign_tol, 1, np.where(dy < -sign_tol, -1, 0))

    nonzero = dy_sign[dy_sign != 0]
    if len(nonzero) == 0:
        monotonicity = 'Flat/None'
    elif np.all(nonzero >= 0) or np.all(nonzero <= 0):
        monotonicity = 'Monotonic'
    else:
        monotonicity = 'Non-monotonic'

    # Direction by endpoint difference normalized by range.
    endpoint_ratio = abs(y_grid[-1] - y_grid[0]) / safe_range(y_grid)
    if endpoint_ratio < 0.05:
        direction = 'None'
    elif y_grid[-1] > y_grid[0]:
        direction = 'Positive'
    else:
        direction = 'Negative'

    # Linearity by deviation from linear fit.
    linearity = 'Linear' if linear_fit_residual_ratio(x_grid, y_grid) < 0.05 else 'Nonlinear'

    # Convexity only most meaningful for monotonic curves, but calculated generally.
    ddy_scale = max(float(np.nanmax(np.abs(ddy))), 1e-12)
    ddy_tol = max(tol, 1e-4 * ddy_scale)
    ddy_sign = np.where(ddy > ddy_tol, 1, np.where(ddy < -ddy_tol, -1, 0))
    ddy_nonzero = ddy_sign[ddy_sign != 0]
    if len(ddy_nonzero) == 0:
        convexity = 'Linear/None'
    elif np.all(ddy_nonzero >= 0):
        convexity = 'Convex'
    elif np.all(ddy_nonzero <= 0):
        convexity = 'Concave'
    else:
        convexity = 'Mixed'

    # Saturation ratio: late slope / early slope.
    n = len(x_grid)
    early = slice(0, max(3, n // 10))
    late = slice(max(0, n - n // 10), n)
    early_slope = float(np.nanmedian(np.abs(dy[early])))
    late_slope = float(np.nanmedian(np.abs(dy[late])))
    slope_ratio = late_slope / max(early_slope, 1e-12)
    saturation = 'Strong' if slope_ratio < 0.1 and early_slope > 1e-8 else 'No/Weak'

    # Turning points by sign changes in dy.
    signs = dy_sign.copy()
    # Fill zeros by nearest nonzero sign to avoid artificial sign changes.
    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]
    for i in range(len(signs) - 2, -1, -1):
        if signs[i] == 0:
            signs[i] = signs[i + 1]
    changes = np.where(np.diff(signs) != 0)[0]
    # Remove adjacent duplicate changes.
    if len(changes) > 1:
        filtered = [changes[0]]
        for c in changes[1:]:
            if c - filtered[-1] > 5:
                filtered.append(c)
        changes = np.array(filtered, dtype=int)

    tp_count_num = int(len(changes))
    if tp_count_num == 0:
        tp_count = 0
    elif tp_count_num == 1:
        tp_count = 1
    elif tp_count_num == 2:
        tp_count = 2
    else:
        tp_count = 'multiple'

    tp_types = []
    for idx in changes:
        left_s = signs[idx]
        right_s = signs[idx + 1] if idx + 1 < len(signs) else signs[idx]
        if left_s > 0 and right_s < 0:
            tp_types.append('peak')
        elif left_s < 0 and right_s > 0:
            tp_types.append('valley')
        else:
            tp_types.append('unknown')

    if len(tp_types) == 0:
        tp_pattern = None
    elif len(tp_types) <= 2:
        tp_pattern = '-'.join(tp_types)
    else:
        tp_pattern = 'alternating'

    return {
        'direction': direction,
        'monotonicity': monotonicity,
        'linearity': linearity,
        'convexity': convexity,
        'saturation': saturation,
        'saturation_late_early_slope_ratio': slope_ratio,
        'tp_count': tp_count,
        'tp_types': tp_types if tp_types else None,
        'tp_pattern': tp_pattern,
        'early_abs_slope': early_slope,
        'late_abs_slope': late_slope,
    }


def strength_label_from_snr(snr: float) -> str:
    if np.isinf(snr):
        return 'Strong'
    if snr < 0.3:
        return 'Very Weak'
    if snr <= 1:
        return 'Weak'
    if snr <= 10:
        return 'Medium'
    return 'Strong'


def sigma_from_snr(function_callable: Callable[[np.ndarray], np.ndarray], snr: float, rng: Optional[np.random.Generator] = None) -> float:
    """Compute constant epsilon sigma from target SNR using x ~ Uniform(0,1)."""
    if np.isinf(snr):
        return 0.0
    x_ref = np.linspace(0.0, 1.0, 10000)
    f_ref = function_callable(x_ref)
    var_f = float(np.nanvar(f_ref))
    if var_f < 1e-12:
        # For null cases, SNR is not meaningful. Use a controlled noise scale.
        # Lower SNR -> larger noise; higher SNR -> smaller noise.
        return float(1.0 / np.sqrt(max(snr, 1e-12)))
    return float(np.sqrt(var_f / snr))


def sample_x(n: int, distribution: str, rng: np.random.Generator) -> np.ndarray:
    if distribution == 'uniform':
        x = rng.uniform(0, 1, n)
    elif distribution == 'beta_2_5':
        x = rng.beta(2, 5, n)
    elif distribution == 'beta_5_2':
        x = rng.beta(5, 2, n)
    elif distribution == 'beta_5_5':
        x = rng.beta(5, 5, n)
    elif distribution == 'two_clusters':
        n1 = n // 2
        x = np.concatenate([rng.uniform(0, 0.3, n1), rng.uniform(0.7, 1.0, n - n1)])
    elif distribution == 'three_clusters':
        n1 = n // 3
        n2 = n // 3
        x = np.concatenate([
            rng.uniform(0, 0.15, n1),
            rng.uniform(0.4, 0.6, n2),
            rng.uniform(0.85, 1.0, n - n1 - n2),
        ])
    else:
        raise ValueError(f'Unknown x distribution: {distribution}')
    return np.sort(x)


def noise_multiplier(x: np.ndarray, structure: str) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if structure == 'constant':
        m = np.ones_like(x)
    elif structure == 'increasing':
        m = 0.2 + 1.6 * x
    elif structure == 'decreasing':
        m = 1.8 - 1.6 * x
    elif structure == 'middle_high':
        m = 0.3 + 1.4 * np.exp(-((x - 0.5) ** 2) / 0.02)
    else:
        raise ValueError(f'Unknown noise structure: {structure}')

    # Normalize so mean multiplier is approximately 1.
    return m / max(float(np.mean(m)), 1e-12)


## 2. Function library F00–F20

Each function family returns approximately 20 shape configurations. The functions are normalized internally where useful, but the final generated y values are not normalized after adding noise.


In [ ]:
# ============================================================
# 2. Function library: F00-F20
# ============================================================

FunctionSpec = Dict[str, Any]


def make_function_specs() -> List[FunctionSpec]:
    specs: List[FunctionSpec] = []

    def add(fid: str, fname: str, formula: str, params: Dict[str, Any], func: Callable[[np.ndarray], np.ndarray], extra: Optional[Dict[str, Any]] = None):
        spec = {
            'function_type': fid,
            'function_name': fname,
            'function_formula': formula,
            'shape_params': params,
            'func': func,
        }
        if extra:
            spec.update(extra)
        specs.append(spec)

    # F00 Random / null baseline: 20 constant-plus-noise settings.
    for i, base in enumerate(np.linspace(0.1, 0.9, 20), start=1):
        add('F00', 'Random', 'f(x)=constant', {'constant': float(base)}, lambda x, base=base: np.full_like(x, base, dtype=float))

    # F01 Linear
    slopes = [-5, -3, -2, -1, -0.5, -0.3, -0.1, 0.1, 0.3, 0.5, 1, 2, 3, 5, 0.01, 0.05, 10, 15, 20, 50]
    for a in slopes:
        add('F01', 'Linear', 'f(x)=a*x', {'a': float(a)}, lambda x, a=a: a * x)

    # F02 Power-Convex
    p_values = [1.1, 1.2, 1.3, 1.5, 1.7, 2.0, 2.3, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0, 8.0, 10.0, 12.0, 15.0, 20.0]
    for p in p_values:
        add('F02', 'Power-Convex', 'f(x)=x^p', {'p': float(p)}, lambda x, p=p: x ** p)

    # F03 Power-Concave
    p_values = [0.95, 0.9, 0.85, 0.8, 0.7, 0.6, 0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.12, 0.1, 0.08, 0.06, 0.04, 0.02]
    for p in p_values:
        add('F03', 'Power-Concave', 'f(x)=x^p', {'p': float(p)}, lambda x, p=p: x ** p)

    # F04 Saturation
    k_values = [0.3, 0.5, 0.8, 1, 1.5, 2, 3, 4, 5, 6, 7, 8, 10, 12, 15, 18, 20, 25, 30, 40]
    for k in k_values:
        add('F04', 'Saturation', 'f(x)=1-exp(-k*x)', {'k': float(k)}, lambda x, k=k: 1.0 - np.exp(-k * x))

    # F05 Log
    a_values = [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 70, 100, 150, 200, 300, 500, 700, 1000]
    for a in a_values:
        add('F05', 'Log', 'f(x)=log(1+a*x)', {'a': float(a)}, lambda x, a=a: np.log1p(a * x))

    # F06 Exponential
    b_values = [0.1, 0.3, 0.5, 0.8, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15]
    for b in b_values:
        add('F06', 'Exponential', 'f(x)=exp(b*x)-1', {'b': float(b)}, lambda x, b=b: np.exp(b * x) - 1.0)

    # F07 S-curve
    scurve_pairs = [
        (3, 0.5), (5, 0.3), (5, 0.5), (5, 0.7),
        (8, 0.3), (8, 0.5), (8, 0.7),
        (12, 0.3), (12, 0.5), (12, 0.7),
        (20, 0.3), (20, 0.5), (20, 0.7),
        (30, 0.5), (50, 0.3), (50, 0.5), (50, 0.7),
        (80, 0.5), (120, 0.5), (200, 0.5),
    ]
    for k, c in scurve_pairs:
        add('F07', 'S-curve', 'f(x)=1/(1+exp(-k*(x-c)))', {'k': float(k), 'c': float(c)}, lambda x, k=k, c=c: 1.0 / (1.0 + np.exp(-k * (x - c))), {'s_curve': True})

    # F08 Threshold
    for gap in [0.5, 1, 2, 5]:
        for delta in [0.005, 0.01, 0.03, 0.05, 0.1]:
            c = 0.5
            a0 = 0.0
            b0 = gap
            def threshold_func(x, gap=gap, delta=delta, c=c, a0=a0, b0=b0):
                x = np.asarray(x)
                y = np.empty_like(x, dtype=float)
                left = x < c - delta
                mid = (x >= c - delta) & (x <= c + delta)
                right = x > c + delta
                y[left] = a0
                y[mid] = a0 + (b0 - a0) / (2 * delta) * (x[mid] - c + delta)
                y[right] = b0
                return y
            add('F08', 'Threshold', 'piecewise threshold', {'gap': float(gap), 'delta': float(delta), 'c': float(c)}, threshold_func, {'threshold': True})

    # F09 Quadratic peak
    quad_pairs = [(a, c) for a in [1, 2, 4, 8] for c in [0.2, 0.35, 0.5, 0.65, 0.8]]
    for a, c in quad_pairs:
        add('F09', 'Quadratic peak', 'f(x)=-a*(x-c)^2', {'a': float(a), 'c': float(c)}, lambda x, a=a, c=c: normalize_range(-a * (x - c) ** 2))

    # F10 Quadratic valley
    for a, c in quad_pairs:
        add('F10', 'Quadratic valley', 'f(x)=a*(x-c)^2', {'a': float(a), 'c': float(c)}, lambda x, a=a, c=c: normalize_range(a * (x - c) ** 2))

    # F11 Spike narrow peak
    for w in [0.01, 0.02, 0.05, 0.1, 0.2]:
        for h in [0.5, 1, 2, 5]:
            c = 0.5
            def spike_func(x, w=w, h=h, c=c):
                x = np.asarray(x)
                y = h * np.maximum(0.0, 1.0 - np.abs(x - c) / (w / 2.0))
                return y
            add('F11', 'Spike', 'narrow triangular peak', {'w': float(w), 'h': float(h), 'c': float(c)}, spike_func)

    # F12 L-shaped narrow valley
    for w in [0.01, 0.02, 0.05, 0.1, 0.2]:
        for h in [0.5, 1, 2, 5]:
            c = 0.5
            def valley_func(x, w=w, h=h, c=c):
                x = np.asarray(x)
                y = -h * np.maximum(0.0, 1.0 - np.abs(x - c) / (w / 2.0))
                return y
            add('F12', 'L-shaped valley', 'narrow triangular valley', {'w': float(w), 'h': float(h), 'c': float(c)}, valley_func)

    # F13/F14 Cubic M/W families: use 20 root configurations.
    root_configs = []
    for spread in [0.3, 0.5, 0.7, 0.9]:
        for offset in [-0.1, 0, 0.1, 0.15, -0.15]:
            center = 0.5 + offset
            r1 = max(0.02, center - spread / 2)
            r2 = center
            r3 = min(0.98, center + spread / 2)
            root_configs.append((r1, r2, r3, spread, offset))
    root_configs = root_configs[:20]
    for r1, r2, r3, spread, offset in root_configs:
        add('F13', 'Cubic M', 'f(x)=(x-r1)(x-r2)(x-r3)', {'r1': float(r1), 'r2': float(r2), 'r3': float(r3)}, lambda x, r1=r1, r2=r2, r3=r3: normalize_range((x-r1)*(x-r2)*(x-r3)))
        add('F14', 'Cubic W', 'f(x)=-(x-r1)(x-r2)(x-r3)', {'r1': float(r1), 'r2': float(r2), 'r3': float(r3)}, lambda x, r1=r1, r2=r2, r3=r3: normalize_range(-(x-r1)*(x-r2)*(x-r3)))

    # F15 Double Gaussian: select first 20 from 24 configs.
    dg_configs = []
    for dist in [0.2, 0.3, 0.4, 0.6]:
        for ratio in [(1,1), (2,1), (1,2)]:
            for sig in [0.05, 0.1]:
                dg_configs.append((dist, ratio[0], ratio[1], sig))
    for dist, A1, A2, sig in dg_configs[:20]:
        mu1 = 0.5 - dist / 2
        mu2 = 0.5 + dist / 2
        add('F15', 'Double Gaussian', 'two gaussian peaks', {'mu1': float(mu1), 'mu2': float(mu2), 'A1': float(A1), 'A2': float(A2), 'sigma': float(sig)}, lambda x, mu1=mu1, mu2=mu2, A1=A1, A2=A2, sig=sig: normalize_range(A1*np.exp(-((x-mu1)**2)/(2*sig**2)) + A2*np.exp(-((x-mu2)**2)/(2*sig**2))))

    # F16 Pure Oscillation
    for omega in [2*np.pi, 3*np.pi, 4*np.pi, 5*np.pi, 6*np.pi, 8*np.pi, 10*np.pi, 12*np.pi, 16*np.pi, 20*np.pi]:
        for A in [0.3, 1.0]:
            add('F16', 'Pure Oscillation', 'f(x)=A*sin(omega*x)', {'omega': float(omega), 'A': float(A)}, lambda x, omega=omega, A=A: A * np.sin(omega * x), {'tp_trend': 'none'})

    # F17 Oscillation + Trend
    for omega in [2*np.pi, 4*np.pi, 6*np.pi, 8*np.pi, 12*np.pi]:
        for b in [-1, -0.3, 0.3, 1]:
            A = 0.3 * abs(b)
            add('F17', 'Oscillation + Trend', 'f(x)=b*x+A*sin(omega*x)', {'omega': float(omega), 'b': float(b), 'A': float(A)}, lambda x, omega=omega, b=b, A=A: b*x + A*np.sin(omega*x), {'tp_trend': 'positive' if b > 0 else 'negative'})

    # F18 Damped Oscillation
    for omega in [4*np.pi, 6*np.pi, 8*np.pi, 12*np.pi, 16*np.pi]:
        for lam in [0.5, 1, 3, 5]:
            A = 1.0
            add('F18', 'Damped Oscillation', 'f(x)=A*exp(-lambda*x)*sin(omega*x)', {'omega': float(omega), 'lambda': float(lam), 'A': float(A)}, lambda x, omega=omega, lam=lam, A=A: A*np.exp(-lam*x)*np.sin(omega*x), {'tp_trend': 'damped'})

    # F19 Growing Oscillation
    for omega in [4*np.pi, 6*np.pi, 8*np.pi, 12*np.pi, 16*np.pi]:
        for lam in [0.3, 0.5, 1, 2]:
            A = 1.0
            add('F19', 'Growing Oscillation', 'f(x)=A*exp(lambda*x)*sin(omega*x)', {'omega': float(omega), 'lambda': float(lam), 'A': float(A)}, lambda x, omega=omega, lam=lam, A=A: A*np.exp(lam*x)*np.sin(omega*x), {'tp_trend': 'growing'})

    # F20 Varying Frequency
    for omega in [3*np.pi, 5*np.pi, 7*np.pi, 10*np.pi, 14*np.pi]:
        for alpha in [0.5, 1, 2, 3]:
            A = 1.0
            add('F20', 'Varying Frequency', 'f(x)=A*sin(omega*x*(1+alpha*x))', {'omega': float(omega), 'alpha': float(alpha), 'A': float(A)}, lambda x, omega=omega, alpha=alpha, A=A: A*np.sin(omega*x*(1 + alpha*x)))

    return specs


FUNCTION_SPECS = make_function_specs()
function_summary = pd.DataFrame(FUNCTION_SPECS).groupby(['function_type', 'function_name']).size().reset_index(name='n_configs')
function_summary


## 3. Metadata and label generation

The metadata table is the main output for downstream experiments. Each row is a case configuration. Realizations can be generated later from any row.


In [ ]:
# ============================================================
# 3. Metadata and label generation
# ============================================================

def make_case_id(function_type: str, shape_index: int, snr: float, noise_structure: str, x_distribution: str) -> str:
    snr_txt = 'inf' if np.isinf(snr) else str(snr).replace('.', 'p')
    return f'{function_type}_shape{shape_index:02d}_SNR{snr_txt}_{noise_structure}_{x_distribution}'


def labels_for_spec(spec: FunctionSpec, snr: float, noise_structure: str, x_distribution: str) -> Dict[str, Any]:
    x_grid = np.linspace(0, 1, 2000)
    y_grid = spec['func'](x_grid)
    labels = derivative_labels(x_grid, y_grid)

    labels['s_curve'] = bool(spec.get('s_curve', False))
    labels['threshold'] = bool(spec.get('threshold', False))
    labels['strength'] = strength_label_from_snr(snr)
    labels['changing_spread'] = noise_structure != 'constant'
    labels['density'] = 'Even' if x_distribution == 'uniform' else 'Uneven'
    labels['clusters'] = x_distribution in ['two_clusters', 'three_clusters']
    labels['high_spread'] = None  # filled after sigma_epsilon is known.

    # Override specific known labels for clarity.
    if spec['function_type'] == 'F00':
        labels.update({
            'direction': 'None',
            'monotonicity': 'Flat/None',
            'linearity': 'None',
            'convexity': 'None',
            'saturation': 'None',
            'tp_count': 0,
            'tp_types': None,
            'tp_pattern': None,
        })
    if 'tp_trend' in spec:
        labels['tp_trend'] = spec['tp_trend']
    else:
        labels['tp_trend'] = None

    return labels


def build_metadata(
    snr_levels: List[float],
    noise_structures: List[str],
    x_distributions: List[str],
    selected_function_types: Optional[List[str]] = None,
    selected_typical_only: bool = False,
    n: int = N,
    r: int = R,
) -> pd.DataFrame:
    records = []
    filtered_specs = FUNCTION_SPECS
    if selected_function_types is not None:
        filtered_specs = [s for s in filtered_specs if s['function_type'] in selected_function_types]

    if selected_typical_only:
        # For each selected function type, take a typical middle configuration.
        typical = []
        for ft in selected_function_types or sorted(set(s['function_type'] for s in filtered_specs)):
            specs_ft = [s for s in filtered_specs if s['function_type'] == ft]
            typical.append(specs_ft[len(specs_ft) // 2])
        filtered_specs = typical

    shape_counter: Dict[str, int] = {}
    for spec in filtered_specs:
        ft = spec['function_type']
        shape_counter.setdefault(ft, 0)
        shape_counter[ft] += 1
        shape_index = shape_counter[ft]

        for snr in snr_levels:
            sigma_eps = sigma_from_snr(spec['func'], snr)
            x_grid = np.linspace(0, 1, 2000)
            f_grid = spec['func'](x_grid)
            f_range = safe_range(f_grid)

            for ns in noise_structures:
                for xd in x_distributions:
                    labels = labels_for_spec(spec, snr, ns, xd)
                    labels['high_spread'] = bool((sigma_eps / f_range) > 0.3) if f_range > 1e-12 else True
                    case_id = make_case_id(ft, shape_index, snr, ns, xd)

                    rec = {
                        'case_id': case_id,
                        'function_type': ft,
                        'function_name': spec['function_name'],
                        'function_formula': spec['function_formula'],
                        'shape_index': shape_index,
                        'shape_params_json': json.dumps(spec['shape_params'], ensure_ascii=False),
                        'snr': 'inf' if np.isinf(snr) else float(snr),
                        'sigma_epsilon': sigma_eps,
                        'noise_structure': ns,
                        'x_distribution': xd,
                        'n': n,
                        'R': r,
                    }
                    rec.update(labels)
                    records.append(rec)

    return pd.DataFrame(records)


# Main experiment: shape x SNR, constant spread, uniform x.
main_metadata = build_metadata(
    snr_levels=SNR_LEVELS,
    noise_structures=['constant'],
    x_distributions=['uniform'],
)

print('Main metadata shape:', main_metadata.shape)
main_metadata.head()


In [ ]:
# Save main metadata
main_metadata_path = OUTPUT_DIR / 'main_experiment_metadata.csv'
main_metadata.to_csv(main_metadata_path, index=False, encoding='utf-8-sig')
print('Saved:', main_metadata_path)
print('Number of main cases:', len(main_metadata))


## 4. Interference experiment metadata

Representative function types are crossed with selected SNR levels and either heteroscedastic noise structures or non-uniform x distributions.


In [ ]:
# ============================================================
# 4. Interference experiment metadata
# ============================================================

REPRESENTATIVE_FUNCTION_TYPES = ['F01', 'F04', 'F07', 'F08', 'F09', 'F13', 'F17', 'F00']

# A: changing noise structure
hetero_metadata = build_metadata(
    snr_levels=INTERFERENCE_SNR_LEVELS,
    noise_structures=['increasing', 'decreasing', 'middle_high'],
    x_distributions=['uniform'],
    selected_function_types=REPRESENTATIVE_FUNCTION_TYPES,
    selected_typical_only=True,
)

# B: x sampling distribution
sampling_metadata = build_metadata(
    snr_levels=INTERFERENCE_SNR_LEVELS,
    noise_structures=['constant'],
    x_distributions=['beta_2_5', 'beta_5_2', 'beta_5_5', 'two_clusters', 'three_clusters'],
    selected_function_types=REPRESENTATIVE_FUNCTION_TYPES,
    selected_typical_only=True,
)

hetero_path = OUTPUT_DIR / 'interference_heteroscedasticity_metadata.csv'
sampling_path = OUTPUT_DIR / 'interference_sampling_metadata.csv'
hetero_metadata.to_csv(hetero_path, index=False, encoding='utf-8-sig')
sampling_metadata.to_csv(sampling_path, index=False, encoding='utf-8-sig')

all_metadata = pd.concat([main_metadata, hetero_metadata, sampling_metadata], ignore_index=True)
all_path = OUTPUT_DIR / 'all_case_metadata.csv'
all_metadata.to_csv(all_path, index=False, encoding='utf-8-sig')

print('Heteroscedasticity cases:', len(hetero_metadata))
print('Sampling-distribution cases:', len(sampling_metadata))
print('All cases:', len(all_metadata))
print('Saved:', all_path)


## 5. Generate scatterplot realizations

The function below generates one realization or many realizations for a given metadata row.

Important: the full design has 2,164,000 scatterplots, so do not generate all realizations unless you really need them.


In [ ]:
# ============================================================
# 5. Realization generation
# ============================================================

def get_spec_by_type_and_index(function_type: str, shape_index: int) -> FunctionSpec:
    specs_ft = [s for s in FUNCTION_SPECS if s['function_type'] == function_type]
    if not (1 <= int(shape_index) <= len(specs_ft)):
        raise ValueError(f'Invalid shape_index={shape_index} for {function_type}')
    return specs_ft[int(shape_index) - 1]


def generate_realization(case_row: pd.Series | Dict[str, Any], realization_index: int = 1) -> Dict[str, np.ndarray]:
    """Generate one realization for one case.

    Random seed follows: seed = numeric_case_hash * 1000 + realization_index.
    Here numeric_case_hash is a stable integer derived from the case_id.
    """
    row = dict(case_row)
    case_id = row['case_id']
    ft = row['function_type']
    shape_index = int(row['shape_index'])
    n = int(row.get('n', N))
    sigma_eps = float(row['sigma_epsilon'])
    noise_structure = row['noise_structure']
    x_distribution = row['x_distribution']

    # Stable deterministic seed from case_id.
    numeric_hash = abs(hash(case_id)) % (2**31 - 1)
    seed = numeric_hash * 1000 + int(realization_index)
    seed = seed % (2**32 - 1)
    rng = np.random.default_rng(seed)

    spec = get_spec_by_type_and_index(ft, shape_index)
    x = sample_x(n, x_distribution, rng)
    f_x = spec['func'](x)
    mult = noise_multiplier(x, noise_structure)
    eps = rng.normal(loc=0.0, scale=sigma_eps * mult, size=n)
    y = f_x + eps

    return {
        'x': x,
        'y': y,
        'f_x_clean': f_x,
        'epsilon': eps,
        'seed': np.array(seed, dtype=np.uint64),
    }


def generate_case_realizations(case_row: pd.Series | Dict[str, Any], R_realizations: int = R) -> Dict[str, np.ndarray]:
    xs, ys, fs = [], [], []
    seeds = []
    for rr in range(1, R_realizations + 1):
        out = generate_realization(case_row, realization_index=rr)
        xs.append(out['x'])
        ys.append(out['y'])
        fs.append(out['f_x_clean'])
        seeds.append(int(out['seed']))
    return {
        'x': np.stack(xs, axis=0),
        'y': np.stack(ys, axis=0),
        'f_x_clean': np.stack(fs, axis=0),
        'seed': np.array(seeds, dtype=np.uint64),
    }


def save_case_realizations(case_row: pd.Series | Dict[str, Any], output_dir: Path = OUTPUT_DIR / 'realizations', R_realizations: int = R) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    row = dict(case_row)
    data = generate_case_realizations(row, R_realizations=R_realizations)
    path = output_dir / f"{row['case_id']}.npz"
    np.savez_compressed(
        path,
        x=data['x'],
        y=data['y'],
        f_x_clean=data['f_x_clean'],
        seed=data['seed'],
        metadata=json.dumps(row, ensure_ascii=False, default=str),
    )
    return path


# Example: generate one realization from one saturation case.
example_row = main_metadata[(main_metadata['function_type'] == 'F04') & (main_metadata['snr'].astype(str) == '5.0')].iloc[0]
example = generate_realization(example_row, realization_index=1)
print(example_row['case_id'])
print(example['x'].shape, example['y'].shape)


## 6. Preview plots

The plotting function is only for checking whether the generated shapes look correct. It does not compute metrics.


In [ ]:
# ============================================================
# 6. Preview plots
# ============================================================

def plot_realization(case_row: pd.Series | Dict[str, Any], realization_index: int = 1, ax: Optional[plt.Axes] = None, show_clean: bool = True):
    row = dict(case_row)
    data = generate_realization(row, realization_index=realization_index)
    if ax is None:
        fig, ax = plt.subplots(figsize=(4.2, 3.2))
    ax.scatter(data['x'], data['y'], s=12, alpha=0.65)
    if show_clean:
        order = np.argsort(data['x'])
        ax.plot(data['x'][order], data['f_x_clean'][order], linewidth=2)
    ax.set_title(row['case_id'], fontsize=8)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    return ax


def preview_function_family(function_type: str, snr: float = 10, n_cols: int = 5, realization_index: int = 1):
    snr_value = 'inf' if np.isinf(snr) else snr
    subset = main_metadata[(main_metadata['function_type'] == function_type) & (main_metadata['snr'].astype(str) == str(snr_value))]
    if subset.empty and not np.isinf(snr):
        subset = main_metadata[(main_metadata['function_type'] == function_type) & (main_metadata['snr'].astype(float) == float(snr))]
    subset = subset.head(20)
    n_rows = math.ceil(len(subset) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.0, n_rows * 2.5), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, row) in zip(axes.ravel(), subset.iterrows()):
        ax.axis('on')
        plot_realization(row, realization_index=realization_index, ax=ax, show_clean=True)
    fig.tight_layout()
    return fig

# Preview examples
fig = preview_function_family('F04', snr=10, n_cols=5, realization_index=1)
plt.show()


## 7. Optional: save a small preview dataset

This cell saves only a few cases. Use it to test your downstream loading code before generating large-scale data.


In [ ]:
# ============================================================
# 7. Optional small data export
# ============================================================

preview_cases = pd.concat([
    main_metadata[main_metadata['function_type'] == 'F01'].head(2),
    main_metadata[main_metadata['function_type'] == 'F04'].head(2),
    main_metadata[main_metadata['function_type'] == 'F08'].head(2),
    main_metadata[main_metadata['function_type'] == 'F16'].head(2),
], ignore_index=True)

preview_dir = OUTPUT_DIR / 'preview_realizations'
preview_dir.mkdir(parents=True, exist_ok=True)

saved_files = []
for _, row in preview_cases.iterrows():
    saved_files.append(save_case_realizations(row, output_dir=preview_dir, R_realizations=3))

print('Saved preview realization files:')
for p in saved_files:
    print(' ', p)


## 8. Optional: controlled full generation

Use the function below only when you are ready to generate a larger dataset. The default is deliberately limited.


In [ ]:
# ============================================================
# 8. Optional controlled full generation
# ============================================================

def generate_and_save_many_cases(
    metadata_df: pd.DataFrame,
    output_dir: Path = OUTPUT_DIR / 'realizations',
    R_realizations: int = R,
    max_cases: Optional[int] = 10,
) -> List[Path]:
    """Generate and save many cases.

    Parameters
    ----------
    metadata_df:
        Case metadata table.
    output_dir:
        Directory for .npz files.
    R_realizations:
        Number of realizations per case.
    max_cases:
        Safety limit. Set to None only when you really want all cases.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    if max_cases is not None:
        metadata_df = metadata_df.head(max_cases)

    paths = []
    for i, (_, row) in enumerate(metadata_df.iterrows(), start=1):
        path = save_case_realizations(row, output_dir=output_dir, R_realizations=R_realizations)
        paths.append(path)
        if i % 10 == 0:
            print(f'Saved {i} cases...')
    print(f'Done. Saved {len(paths)} case files to {output_dir}')
    return paths

# Example, deliberately small:
# paths = generate_and_save_many_cases(main_metadata, R_realizations=5, max_cases=20)


## 9. Quick checks

These checks confirm that the expected case counts are produced.


In [ ]:
# ============================================================
# 9. Quick checks
# ============================================================

print('Function configs:', len(FUNCTION_SPECS))
print('Main cases:', len(main_metadata))
print('Heteroscedasticity interference cases:', len(hetero_metadata))
print('Sampling interference cases:', len(sampling_metadata))
print('All cases:', len(all_metadata))

assert len(FUNCTION_SPECS) == 420
assert len(main_metadata) == 21000
assert len(hetero_metadata) == 240
assert len(sampling_metadata) == 400
assert len(all_metadata) == 21640

main_metadata.groupby('function_type').size().head()
